In [ ]:
%%sql -r dataframe_2
CREATE DATABASE IF NOT EXISTS MCP_DB;

-- Step 2: Create Schema
CREATE SCHEMA IF NOT EXISTS MCP_DB.MCP_SCHEMA;

-- Step 3: Use the DB and Schema
USE DATABASE MCP_DB;
USE SCHEMA MCP_SCHEMA;

In [ ]:
%%sql -r dataframe_1
-- Step 5: Create Employee Table
CREATE OR REPLACE TABLE MCP_DB.MCP_SCHEMA.EMPLOYEE (
    EMPLOYEE_ID  INT AUTOINCREMENT PRIMARY KEY,
    FIRST_NAME   VARCHAR(50),
    LAST_NAME    VARCHAR(50),
    DEPARTMENT   VARCHAR(50),
    JOB_TITLE    VARCHAR(100),
    SALARY       NUMBER(10,2),
    JOINING_DATE DATE,
    LOCATION     VARCHAR(50),
    EMAIL        VARCHAR(100)
);

-- Step 5: Insert Sample Data
INSERT INTO MCP_DB.MCP_SCHEMA.EMPLOYEE (
    FIRST_NAME, LAST_NAME, DEPARTMENT, JOB_TITLE,
    SALARY, JOINING_DATE, LOCATION, EMAIL
)
VALUES
    ('Rahul',  'Sharma',  'Engineering', 'Software Engineer',        85000,  '2021-03-15', 'Hyderabad', 'rahul.sharma@company.com'),
    ('Priya',  'Patel',   'Engineering', 'Senior Software Engineer', 110000, '2019-07-01', 'Bangalore', 'priya.patel@company.com'),
    ('Amit',   'Verma',   'Sales',       'Sales Executive',          60000,  '2022-01-10', 'Mumbai',    'amit.verma@company.com'),
    ('Sneha',  'Reddy',   'Sales',       'Sales Manager',            90000,  '2020-05-20', 'Hyderabad', 'sneha.reddy@company.com'),
    ('Vikram', 'Singh',   'HR',          'HR Executive',             55000,  '2023-02-28', 'Delhi',     'vikram.singh@company.com'),
    ('Anjali', 'Gupta',   'HR',          'HR Manager',               80000,  '2018-11-05', 'Delhi',     'anjali.gupta@company.com'),
    ('Ravi',   'Kumar',   'Finance',     'Financial Analyst',        75000,  '2021-09-12', 'Mumbai',    'ravi.kumar@company.com'),
    ('Deepa',  'Nair',    'Finance',     'Finance Manager',          95000,  '2017-06-30', 'Bangalore', 'deepa.nair@company.com'),
    ('Suresh', 'Iyer',    'Engineering', 'DevOps Engineer',          92000,  '2020-03-22', 'Pune',      'suresh.iyer@company.com'),
    ('Kavya',  'Menon',   'Marketing',   'Marketing Specialist',     65000,  '2022-08-15', 'Bangalore', 'kavya.menon@company.com'),
    ('Arun',   'Pillai',  'Marketing',   'Marketing Manager',        88000,  '2019-04-10', 'Mumbai',    'arun.pillai@company.com'),
    ('Nisha',  'Joshi',   'Engineering', 'QA Engineer',              70000,  '2023-01-05', 'Hyderabad', 'nisha.joshi@company.com'),
    ('Manoj',  'Tiwari',  'Sales',       'Sales Representative',     52000,  '2023-06-20', 'Delhi',     'manoj.tiwari@company.com'),
    ('Pooja',  'Desai',   'Finance',     'Accountant',               62000,  '2022-03-18', 'Pune',      'pooja.desai@company.com'),
    ('Kiran',  'Rao',     'Engineering', 'Data Engineer',            98000,  '2020-11-25', 'Bangalore', 'kiran.rao@company.com');



In [ ]:
%%sql -r dataframe_3
-- Step 6: Verify the data
SELECT * FROM MCP_DB.MCP_SCHEMA.EMPLOYEE;

In [ ]:
%%sql -r dataframe_4
-- Step 7: Create Stage for Semantic Model YAML
CREATE STAGE IF NOT EXISTS MCP_DB.MCP_SCHEMA.MCP_STAGE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Stage for storing Cortex Analyst Semantic Model YAML files';

In [ ]:
%%sql -r dataframe_29
USE ROLE ACCOUNTADMIN;
-- Grant Cortex Analyst access to your role (e.g., SYSADMIN or custom role)
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE ACCOUNTADMIN;
-- Or, for tighter control, use CORTEX_ANALYST_USER (Analyst only, not all AI features)
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_ANALYST_USER TO ROLE SYSADMIN;
-- Make sure your role can use the DB and Schema
GRANT USAGE ON DATABASE MCP_DB TO ROLE ACCOUNTADMIN;
GRANT USAGE ON SCHEMA MCP_DB.MCP_SCHEMA TO ROLE ACCOUNTADMIN;
GRANT SELECT ON TABLE MCP_DB.MCP_SCHEMA.EMPLOYEE TO ROLE ACCOUNTADMIN;
GRANT CREATE SEMANTIC VIEW ON SCHEMA MCP_DB.MCP_SCHEMA TO ROLE ACCOUNTADMIN;

In [ ]:
%%sql -r dataframe_31
-- Show all semantic views in your schema
SHOW SEMANTIC VIEWS IN SCHEMA MCP_DB.MCP_SCHEMA;


In [ ]:
%%sql -r dataframe_5
CREATE OR REPLACE MCP SERVER MCP_DB.MCP_SCHEMA.EMPLOYEE_MCP_SERVER
FROM SPECIFICATION $$
tools:
  - name: "employee-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "MCP_DB.MCP_SCHEMA.EMPLOYEE_SEMANTIC_MODEL"
    description: "Answers questions about employee data including salary, department, location, and job titles"
    title: "Employee Analyst"
  - name: "sql-exec-tool"
    type: "SYSTEM_EXECUTE_SQL"
    title: "SQL Execution Tool"
    description: "Executes SQL queries directly against the Snowflake Employee database"
$$;

In [ ]:
%%sql -r dataframe_6
SHOW MCP SERVERS IN SCHEMA MCP_DB.MCP_SCHEMA;


In [ ]:
%%sql -r dataframe_7
DESCRIBE MCP SERVER MCP_DB.MCP_SCHEMA.EMPLOYEE_MCP_SERVER;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE SECURITY INTEGRATION EMPLOYEE_MCP_OAUTH
  TYPE = OAUTH
  OAUTH_CLIENT = CUSTOM
  ENABLED = TRUE
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'http://localhost:8501'
  OAUTH_ALLOW_NON_TLS_REDIRECT_URI = TRUE;

In [ ]:
%%sql -r dataframe_9
-- Note: integration name must be UPPERCASE
SELECT SYSTEM$SHOW_OAUTH_CLIENT_SECRETS('EMPLOYEE_MCP_OAUTH');

In [ ]:
%%sql -r dataframe_10
-- Grant USAGE on MCP Server to your role
GRANT USAGE ON MCP SERVER MCP_DB.MCP_SCHEMA.EMPLOYEE_MCP_SERVER TO ROLE ACCOUNTADMIN;
-- Grant SELECT on the Semantic View (needed for Cortex Analyst tool)
GRANT SELECT ON SEMANTIC VIEW MCP_DB.MCP_SCHEMA.EMPLOYEE_SEMANTIC_MODEL TO ROLE ACCOUNTADMIN;
-- Grant SELECT on the base Employee table (needed for SQL tool)
GRANT SELECT ON TABLE MCP_DB.MCP_SCHEMA.EMPLOYEE TO ROLE ACCOUNTADMIN;


In [ ]:
import requests
# Your credentials
client_id = "<YOUR_CLIENT_ID>"
client_secret = "<YOUR_CLIENT_SECRET>"
account = "SXPZYRW-ENB74981"
# Get token
url = f"https://{account}.snowflakecomputing.com/oauth/token"
response = requests.post(
    url,
    data={
        "grant_type": "client_credentials",
        "client_id": "nC4gIpsY2+JZUPidaXuA1wPCT14=",
        "client_secret": "HL1yVUuSeN7x4kEjDvwfUAVoMiAvegqKjnrJdb9+2D4="
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)
token_data = response.json()
print("Access Token:", token_data.get("access_token"))
print("Expires In:", token_data.get("expires_in"), "seconds")

```
> Replace `SYSADMIN` with whatever role your VS Code app will use.
---
## 📌 Step 4: Get Your MCP Server Endpoint URL
The MCP server endpoint follows this pattern:
```
https://SXPZYRW-ENB74981.snowflakecomputing.com/api/v2/databases/MCP_DB/schemas/MCP_SCHEMA/mcp-servers/EMPLOYEE_MCP_SERVER
```
Find your account URL in Snowsight:
- Go to your account name (bottom left) → Copy account identifier
- Format: `https://<orgname>-<accountname>.snowflakecomputing.com`
> ⚠️ Use **hyphens (-)** not underscores in the hostname — this is a known MCP connection issue!
---
## 📌 Step 5: Set Up VS Code Project
**Folder structure:**
```
employee-mcp-app/
├── app.py
├── .env
└── requirements.txt
```
**requirements.txt:**
```
streamlit
requests
python-dotenv
snowflake-connector-python

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
token = session.connection.rest.token
print("Access Token:", token)